In [1]:
import numpy as np
import  pandas as pd
import librosa, librosa.display
import matplotlib.pyplot as plt
import os
import torch, torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import IPython.display as ipd
import random
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit

# Data Loading Pipeline

In [5]:
BASE = "/kaggle/input/competitions/birdclef-2025"  # <-- CHANGE THIS to what FOUND printed
# example: BASE = "/kaggle/input/birdclef-2025/birdclef-2025" if needed

train = pd.read_csv(f"{BASE}/train.csv")
taxonomy = pd.read_csv(f"{BASE}/taxonomy.csv")
print(BASE, train.shape)

/kaggle/input/competitions/birdclef-2025 (28564, 13)


In [6]:
taxonomy.head()

,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1139490,1139490,Ragoniella pulchella,Ragoniella pulchella,Insecta
1,1192948,1192948,Oxyprora surinamensis,Oxyprora surinamensis,Insecta
2,1194042,1194042,Copiphora colombiae,Copiphora colombiae,Insecta
3,126247,126247,Leptodactylus insularum,Spotted Foam-nest Frog,Amphibia
4,1346504,1346504,Neoconocephalus brachypterus,Neoconocephalus brachypterus,Insecta


In [7]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
sample_df=train.sample(frac=0.1,random_state=42)
sample_df.shape

(2856, 13)

In [9]:
labels=sorted(train['primary_label'].unique())
label2idx={l:i for i,l in enumerate(labels)}
idx2label={i:l for l,i in label2idx.items()}

In [13]:
class CFG:
    SR=32000
    DURATION=5
    N_MELS=128
    BATCH=32

In [16]:
class BirdDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR, n_mels=CFG.N_MELS, f_min=50, f_max=14000, n_fft=1024, hop_length=512
        )
        self.db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        path = os.path.join(BASE, "train_audio", row['filename'])
        wav,sr=torchaudio.load(path)
        wav=wav.mean(dim=0)
        if sr != CFG.SR:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=CFG.SR)
            wav = resampler(wav)
        target = CFG.SR * CFG.DURATION
        if len(wav) > target:
            start = torch.randint(0, len(wav)-target, (1,)).item()
            wav = wav[start:start+target]
        else:
            wav = F.pad(wav, (0, target - len(wav)))
            
        mel = self.mel(wav)
        mel = self.db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        
        label = label2idx[row['primary_label']]
        return mel, label


In [17]:
splitter=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)

In [18]:
counts=sample_df['primary_label'].value_counts()
counts

primary_label
grekis     101
compau      90
roahaw      63
trokin      63
wbwwre1     60
          ... 
46010        1
plctan1      1
65419        1
22333        1
1462737      1
Name: count, Length: 178, dtype: int64

In [19]:
rare_classes=counts[counts==1].index
rare_classes.shape

(21,)

In [20]:
sample_df = sample_df[~sample_df['primary_label'].isin(rare_classes)].reset_index(drop=True)

In [22]:
sample_df.shape

(2835, 13)

In [23]:
train_idx, val_idx = next(splitter.split(sample_df, sample_df['primary_label']))

In [24]:
train_df = sample_df.iloc[train_idx].reset_index(drop=True)
val_df = sample_df.iloc[val_idx].reset_index(drop=True)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")

Train size: 2268 | Val size: 567


In [25]:
train_dataset=BirdDataset(train_df)
val_dataset=BirdDataset(val_df)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [27]:
mels, labs = next(iter(train_loader))
print("Batch shape:", mels.shape)       
print("Labels shape:", labs.shape)

Batch shape: torch.Size([32, 128, 313])
Labels shape: torch.Size([32])


# Buidling Hybird Model

In [29]:
import torch.nn as nn
import timm

class HybridBirdModel(nn.Module):
    def __init__(self, num_classes=206):
        super().__init__()
        
        # Step A: Backbone (CNN feature extractor)
        self.backbone = timm.create_model(
            'tf_efficientnet_b0', 
            pretrained=True, 
            in_chans=1, 
            num_classes=0, 
            global_pool=''
        )
        
        # Step C: Transformer (Processes the sequence of patches)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=1280, 
            nhead=8, 
            dropout=0.1, 
            batch_first=True # Crucial: PyTorch expects [Batch, Seq, Features]
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Step D: Final Classifier
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        # x is [B, 1, 128, 313]
        
        # 1. CNN Backbone
        # Output: [B, 1280, 4, 10]
        features = self.backbone(x)
        
        # 2. Step B: Flattening Magic
        # Change [B, 1280, 4, 10] -> [B, 4, 10, 1280]
        features = features.permute(0, 2, 3, 1) 
        
        # Flatten the spatial dims -> [B, 40, 1280]
        features = features.flatten(1, 2) 
        
        # 3. Transformer
        # Output: [B, 40, 1280]
        trans_out = self.transformer(features)
        
        # 4. Mean Pooling
        # Average across the 40 patches -> [B, 1280]
        pooled = trans_out.mean(dim=1)
        
        # 5. Final Classifier
        # Output: [B, 206]
        logits = self.classifier(pooled)
        
        return logits

In [30]:
model = HybridBirdModel(num_classes=206).to(device)
dummy = torch.randn(2, 1, 128, 313).to(device)

with torch.no_grad():
    out = model(dummy)

print(out.shape) # MUST PRINT: torch.Size([2, 206])

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

torch.Size([2, 206])
